# Live Demo — Clean & Build Panel
### Session 5 · ExInt II · WU Vienna · SS 2026

**What we do in this notebook:**
1. Find the most recent pull folder automatically
2. Read and concatenate all parquet chunks
3. Inspect the raw combined data
4. Drop duplicates and rows missing panel identifiers
5. Convert numeric columns
6. Sort into a firm-year panel
7. Save as `data/processed/panel_clean.parquet`

This is the **mini version** of `02_clean.py` — same logic, using the 5-firm demo data from the pull notebook.

> **Reference project:** https://github.com/vkiefner/sme-intl

---
## Cell 1 — Imports

In [2]:
import os
from pathlib import Path
from dotenv import load_dotenv
from pathlib import Path
from datetime import datetime
import pandas as pd

def find_env():
    current = Path(os.getcwd())
    for path in [current] + list(current.parents):
        if (path / ".env").exists():
            return path / ".env"
        for sibling in path.iterdir():
            if sibling.is_dir() and (sibling / ".env").exists():
                return sibling / ".env"
    raise FileNotFoundError("Could not find .env anywhere.")

env_file = find_env()
project_root = env_file.parent
os.chdir(project_root)

print(f"Working directory: {os.getcwd()}")
print(f"pandas: {pd.__version__}")

Working directory: /Users/thomasmacbookair/Documents/medtech-internationalization-drivers
pandas: 2.2.3


---
## Cell 2 — Find the most recent pull folder

Folders are named `YYYY-MM-DD_HH-MM-SS` so alphabetical sort = chronological sort.  
We always pick the last one — no manual path editing needed.

In [3]:
RAW_DIR  = Path("data") / "raw"
PROC_DIR = Path("data") / "processed"
PROC_DIR.mkdir(parents=True, exist_ok=True)

# List all timestamped pull folders
folders = sorted([f for f in RAW_DIR.iterdir() if f.is_dir()])

print("Pull folders found:")
for f in folders:
    files = list(f.glob("fyear_*.parquet"))
    print(f"  {f.name}  ({len(files)} parquet files)")

# Pick the most recent
latest = folders[-1]
print(f"\nUsing: {latest.name}")

Pull folders found:
  2026-05-29_18-35-39  (11 parquet files)

Using: 2026-05-29_18-35-39


---
## Cell 3 — Read all parquet chunks

In [4]:
parquet_files = sorted(latest.glob("fyear_*.parquet"))
print(f"Reading {len(parquet_files)} files...\n")

chunks = []
for f in parquet_files:
    chunk = pd.read_parquet(f)
    chunks.append(chunk)
    print(f"  {f.name:<25}  {len(chunk):>4} rows")

# Concatenate into one DataFrame
df = pd.concat(chunks, ignore_index=True)
print(f"\nCombined: {df.shape[0]} rows × {df.shape[1]} columns")

Reading 11 files...

  fyear_2015.parquet            5 rows
  fyear_2016.parquet            5 rows
  fyear_2017.parquet            5 rows
  fyear_2018.parquet            5 rows
  fyear_2019.parquet            5 rows
  fyear_2020.parquet            5 rows
  fyear_2021.parquet            5 rows
  fyear_2022.parquet            5 rows
  fyear_2023.parquet            5 rows
  fyear_2024.parquet            5 rows
  fyear_2025.parquet            5 rows

Combined: 55 rows × 444 columns


---
## Cell 4 — First look at the combined data

In [5]:
df.head(10)

,gvkey,indfmt,datafmt,consol,popsrc,acctstd,acqmeth,bspr,compst,curcd,...,conm,costat,fic,loc,naicsh,sich,rank,au,auop,hiid
0,101317,INDL,HIST_STD,C,I,DI,<NA>,GO,<NA>,USD,...,SMITH & NEPHEW PLC,A,GBR,GBR,339113,3842,1,6,1,01W
1,208821,INDL,HIST_STD,C,I,DI,<NA>,GO,<NA>,EUR,...,SARTORIUS AG,A,DEU,DEU,334516,3826,1,6,1,02W
2,221806,INDL,HIST_STD,C,I,DI,<NA>,GO,<NA>,SEK,...,GETINGE AB PUBL,A,SWE,SWE,334510,3845,1,9,1,01W
3,238442,INDL,HIST_STD,C,I,DI,<NA>,GO,<NA>,EUR,...,CARL ZEISS MEDITEC AG,A,DEU,DEU,334510,3845,1,4,1,01W
4,326765,INDL,HIST_STD,C,I,DI,<NA>,GO,<NA>,EUR,...,SIEMENS HEALTHINEE,A,DEU,DEU,334510,3845,1,4,1,<NA>
5,101317,INDL,HIST_STD,C,I,DI,<NA>,GO,<NA>,USD,...,SMITH & NEPHEW PLC,A,GBR,GBR,339113,3842,1,6,1,01W
6,208821,INDL,HIST_STD,C,I,DI,<NA>,GO,<NA>,EUR,...,SARTORIUS AG,A,DEU,DEU,334516,3826,1,6,1,02W
7,221806,INDL,HIST_STD,C,I,DI,<NA>,GO,<NA>,SEK,...,GETINGE AB PUBL,A,SWE,SWE,334510,3845,1,9,1,01W
8,238442,INDL,HIST_STD,C,I,DI,<NA>,GO,<NA>,EUR,...,CARL ZEISS MEDITEC AG,A,DEU,DEU,334510,3845,1,4,1,01W
9,326765,INDL,HIST_STD,C,I,DI,<NA>,GO,<NA>,EUR,...,SIEMENS HEALTHINEE,A,DEU,DEU,334510,3845,1,4,1,<NA>


In [6]:
# Data types overview
print(f"Shape: {df.shape}")
print(f"\nDtype counts:")
print(df.dtypes.value_counts())

Shape: (55, 444)

Dtype counts:
string[python]    279
Float64           153
Int64              11
datetime64[ns]      1
Name: count, dtype: int64


In [7]:
# Panel coverage — firms × years
df.groupby(["gvkey", "conm"])["fyear"].agg(["min", "max", "count"]).rename(
    columns={"min": "first_year", "max": "last_year", "count": "n_years"}
)

,,first_year,last_year,n_years
gvkey,conm,,,
101317,SMITH & NEPHEW PLC,2015,2025,11
208821,SARTORIUS AG,2015,2025,11
221806,GETINGE AB PUBL,2015,2025,11
238442,CARL ZEISS MEDITEC AG,2015,2025,11
326765,SIEMENS HEALTHINEE,2015,2025,11


---
## Cell 5 — Check for duplicate rows

In [8]:
n_dupes = df.duplicated().sum()
print(f"Exact duplicate rows: {n_dupes}")

# Also check: are there duplicate gvkey-fyear combinations?
n_key_dupes = df.duplicated(subset=["gvkey", "fyear"]).sum()
print(f"Duplicate gvkey-fyear pairs: {n_key_dupes}")

# Drop exact duplicates
df = df.drop_duplicates()
print(f"\nAfter dedup: {len(df)} rows")

Exact duplicate rows: 0
Duplicate gvkey-fyear pairs: 0

After dedup: 55 rows


---
## Cell 6 — Drop rows missing panel identifiers

Every row in a firm-year panel **must** have a firm identifier (`gvkey`) and a time identifier (`fyear`).  
Without both, we cannot place the observation in the panel.

In [9]:
n_before = len(df)

df = df.dropna(subset=["gvkey", "fyear"])

n_dropped = n_before - len(df)
print(f"Dropped {n_dropped} rows missing gvkey or fyear")
print(f"Remaining: {len(df)} rows")

Dropped 0 rows missing gvkey or fyear
Remaining: 55 rows


---
## Cell 7 — Convert object columns to numeric where possible

WRDS sometimes returns numeric fields as `object` (string) dtype.  
We use `pd.to_numeric(..., errors='coerce')` — this converts what it can and turns non-numeric values into `NaN` (never crashes).

In [10]:
# Columns we know are strings — leave these alone
STRING_COLS = {
    "gvkey", "conm", "cusip", "isin", "sedol", "tic",
    "naics", "sic", "loc", "curcd", "fic", "exchg",
    "costat", "stalt", "datafmt", "indfmt", "popsrc", "consol"
}

converted = []
for col in df.columns:
    if df[col].dtype == object and col not in STRING_COLS:
        df[col] = pd.to_numeric(df[col], errors="coerce")
        converted.append(col)

print(f"Converted {len(converted)} columns to numeric dtype")
if converted:
    print("Columns converted:", converted)

Converted 0 columns to numeric dtype


---
## Cell 8 — Ensure `fyear` is integer

In [11]:
df["fyear"] = df["fyear"].astype(int)
print(f"fyear dtype: {df['fyear'].dtype}")
print(f"fyear range: {df['fyear'].min()} – {df['fyear'].max()}")

fyear dtype: int64
fyear range: 2015 – 2025


---
## Cell 9 — Sort into a firm-year panel

In [12]:
df = df.sort_values(["gvkey", "fyear"]).reset_index(drop=True)

print("Sorted by gvkey, fyear. First 10 rows:")
df[["gvkey", "conm", "fyear", "loc", "at", "sale", "ib", "xrd", "emp"]].head(10)

Sorted by gvkey, fyear. First 10 rows:


,gvkey,conm,fyear,loc,at,sale,ib,xrd,emp
0,101317,SMITH & NEPHEW PLC,2015,GBR,7167.0,4634.0,410.0,222.0,15.644
1,101317,SMITH & NEPHEW PLC,2016,GBR,7344.0,4669.0,784.0,278.0,15.644
2,101317,SMITH & NEPHEW PLC,2017,GBR,7866.0,4765.0,767.0,229.0,15.933
3,101317,SMITH & NEPHEW PLC,2018,GBR,8059.0,4904.0,663.0,270.0,16.377
4,101317,SMITH & NEPHEW PLC,2019,GBR,9299.0,5138.0,600.0,320.0,17.637
5,101317,SMITH & NEPHEW PLC,2020,GBR,11012.0,4560.0,448.0,344.0,17.914
6,101317,SMITH & NEPHEW PLC,2021,GBR,10920.0,5212.0,524.0,397.0,18.369
7,101317,SMITH & NEPHEW PLC,2022,GBR,9966.0,5215.0,223.0,396.0,19.012
8,101317,SMITH & NEPHEW PLC,2023,GBR,9987.0,5549.0,263.0,385.0,18.452
9,101317,SMITH & NEPHEW PLC,2024,GBR,10354.0,5810.0,412.0,343.0,17.349


---
## Cell 10 — Missing value summary

In [13]:
# How complete is each column?
missing = pd.DataFrame({
    "missing_n":   df.isnull().sum(),
    "missing_pct": (df.isnull().sum() / len(df) * 100).round(1),
    "dtype":       df.dtypes
})

# Show columns with at least one missing value, sorted by % missing
missing[missing["missing_n"] > 0].sort_values("missing_pct", ascending=False)

,missing_n,missing_pct,dtype
acqmeth,55,100.0,string[python]
oancfd,55,100.0,string[python]
pliach,55,100.0,string[python]
pcl,55,100.0,string[python]
pacqp,55,100.0,string[python]
...,...,...,...
invch,3,5.5,Float64
emp,2,3.6,Float64
exre,1,1.8,Float64
txop,1,1.8,Float64


In [15]:
# How complete are the key variables students will use?
key_vars = ["at", "sale", "ib", "xrd", "emp", "dltt", "pifo"]
available_key = [v for v in key_vars if v in df.columns]

print("Completeness of key variables:")
for col in available_key:
    pct_complete = (df[col].notna().sum() / len(df) * 100)
    bar = "█" * int(pct_complete / 5)
    print(f"  {col:<6}  {pct_complete:>5.1f}%  {bar}")

Completeness of key variables:
  at      100.0%  ████████████████████
  sale    100.0%  ████████████████████
  ib      100.0%  ████████████████████
  xrd     100.0%  ████████████████████
  emp      96.4%  ███████████████████
  dltt    100.0%  ████████████████████


---
## Cell 11 — Panel statistics

In [16]:
print("="*45)
print("Panel Statistics")
print("="*45)
print(f"  Total firm-years:   {len(df):>8,}")
print(f"  Unique firms:       {df['gvkey'].nunique():>8,}")
print(f"  Years covered:      {df['fyear'].min()}–{df['fyear'].max()}")
print(f"  Countries:          {df['loc'].nunique():>8,}")
print(f"  Total columns:      {df.shape[1]:>8,}")

print("\n  Observations per year:")
for year, count in df.groupby("fyear").size().items():
    print(f"    {year}: {count:>4}")

Panel Statistics
  Total firm-years:         55
  Unique firms:              5
  Years covered:      2015–2025
  Countries:                 3
  Total columns:           444

  Observations per year:
    2015:    5
    2016:    5
    2017:    5
    2018:    5
    2019:    5
    2020:    5
    2021:    5
    2022:    5
    2023:    5
    2024:    5
    2025:    5


---
## Cell 12 — Quick descriptive stats on key variables

In [17]:
df[available_key].describe().round(2)

,at,sale,ib,xrd,emp,dltt
count,55.0,55.0,55.0,55.0,53.0,55.0
mean,19891.23,11112.37,843.4,759.69,19.52,4060.95
std,19831.86,10993.59,867.46,687.34,20.15,4373.6
min,1139.29,1040.06,-967.0,59.13,2.89,3.0
25%,3213.06,2077.71,158.25,207.5,7.5,702.17
50%,9987.0,5138.0,524.0,373.56,12.83,2848.0
75%,42757.0,21697.0,1400.5,1333.0,17.64,5106.5
max,63918.0,34969.0,3239.0,2160.0,72.0,16006.0


---
## Cell 13 — Save clean panel as parquet

In [18]:
out_path = PROC_DIR / "panel_clean.parquet"

df.to_parquet(out_path, index=False)

size_mb = out_path.stat().st_size / 1_048_576
print(f"Saved: {out_path}")
print(f"Size:  {size_mb:.2f} MB")
print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")

Saved: data/processed/panel_clean.parquet
Size:  0.24 MB
Shape: 55 rows × 444 columns


---
## Cell 14 — Verify: read back and confirm

In [19]:
check = pd.read_parquet(out_path)

print(f"Read back: {check.shape[0]} rows × {check.shape[1]} columns")
print(f"dtypes preserved: {check.dtypes.value_counts().to_dict()}")
check[["gvkey", "conm", "fyear", "at", "sale", "ib"]].head(10)

Read back: 55 rows × 444 columns
dtypes preserved: {string[python]: 279, Float64Dtype(): 153, Int64Dtype(): 10, dtype('int64'): 1, dtype('<M8[ns]'): 1}


,gvkey,conm,fyear,at,sale,ib
0,101317,SMITH & NEPHEW PLC,2015,7167.0,4634.0,410.0
1,101317,SMITH & NEPHEW PLC,2016,7344.0,4669.0,784.0
2,101317,SMITH & NEPHEW PLC,2017,7866.0,4765.0,767.0
3,101317,SMITH & NEPHEW PLC,2018,8059.0,4904.0,663.0
4,101317,SMITH & NEPHEW PLC,2019,9299.0,5138.0,600.0
5,101317,SMITH & NEPHEW PLC,2020,11012.0,4560.0,448.0
6,101317,SMITH & NEPHEW PLC,2021,10920.0,5212.0,524.0
7,101317,SMITH & NEPHEW PLC,2022,9966.0,5215.0,223.0
8,101317,SMITH & NEPHEW PLC,2023,9987.0,5549.0,263.0
9,101317,SMITH & NEPHEW PLC,2024,10354.0,5810.0,412.0


---
## Summary — What just happened

| Step | What we did | Why |
|------|-------------|-----|
| Found latest folder | `sorted(iterdir())[-1]` | No manual path editing — reproducible |
| Read chunks | `pd.read_parquet()` per file | Fast, type-safe |
| Concatenated | `pd.concat()` | One DataFrame for the whole panel |
| Dropped duplicates | `drop_duplicates()` | Avoid inflated row counts |
| Dropped missing IDs | `dropna(subset=[gvkey, fyear])` | Panel must be identifiable |
| Converted dtypes | `pd.to_numeric(..., errors='coerce')` | Numbers stored as numbers |
| Sorted | `sort_values([gvkey, fyear])` | Standard panel structure |
| Saved | `to_parquet()` | Compressed, fast, type-safe |

**Your full script `02_clean.py` does exactly this** — the only difference is it runs on all global firms, not just 5.

**Next session:** we use this panel in `03_descriptives.py` to build summary statistics and figures.

---
*ExInt II · WU Vienna · SS 2026 · github.com/vkiefner/sme-intl*